# Direct Preference Optimization (DPO): The Mathematical Elimination of Reinforcement Learning Loops in LLM Alignment

> **Topic**: LLM Alignment, RLHF, Bradley-Terry Model Reparameterization  
> **Key Breakthrough**: Proving that the optimal policy under a reward model can be solved analytically in closed form.

---

## 1. The Bradley-Terry Preference Model
Given prompt $x$ and pair of responses $(y_w, y_l)$ where human prefers $y_w \succ y_l$:
$$P(y_w \succ y_l | x) = \sigma(r(x, y_w) - r(x, y_l))$$

In classical RLHF, one maximizes:
$$\max_\pi \mathbb{E}_{x, y \sim \pi}[r(x, y)] - \beta \mathbb{D}_{\text{KL}}(\pi(y|x) \parallel \pi_{\text{ref}}(y|x))$$

### The DPO Closed-Form Derivation (Rafailov et al., 2023)
The exact mathematical solution for optimal policy $\pi^*$ satisfies:
$$\pi^*(y|x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y|x) \exp\left( \frac{1}{\beta} r(x, y) \right)$$

Rearranging to solve for the ground-truth reward $r(x, y)$:
$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

Substituting this identity directly into the Bradley-Terry preference probability cancels the partition function $Z(x)$:
$$P(y_w \succ y_l | x) = \sigma\left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right)$$

The DPO loss trains the policy directly via binary classification without a reward model or PPO!


In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(pi_logps_w, pi_logps_l, ref_logps_w, ref_logps_l, beta=0.1):
    """Direct Preference Optimization loss computation."""
    pi_ratio = pi_logps_w - pi_logps_l
    ref_ratio = ref_logps_w - ref_logps_l
    logits = beta * (pi_ratio - ref_ratio)
    return -F.logsigmoid(logits).mean()

# Test synthetic logits
loss = dpo_loss(torch.tensor([2.5]), torch.tensor([1.0]), torch.tensor([1.2]), torch.tensor([1.1]))
print(f"Computed DPO Loss: {loss.item():.4f}")
